# Medical Imaging — 10-Model Weighted Ensemble (Kaggle GPU)

Trains the **same ten algorithms** used for the tabular diseases, on **images**,
for both **brain tumour MRI** and **chest X-ray pneumonia**. Attach one dataset
or both; the notebook trains whatever it finds.

### How images get to the ten algorithms

The ten classifiers cannot read pixels, so a CNN turns each image into a vector
first:

    image  ->  fine-tuned EfficientNetB0  ->  1280-d embedding  ->  PCA 128
           ->  10 classifiers  ->  accuracy-weighted vote  ->  verdict

The backbone is **fine-tuned on the medical images first**, not left frozen.
Frozen ImageNet features are tuned for cats and cars; a few epochs of
fine-tuning moves them toward tissue texture, and the ten heads get a far more
separable space to work in.

Weighting is identical to the tabular side:

    final_probability = Σ ( weightₘ × suretyₘ ) / Σ weightₘ
    weightₘ           = max( cv_accuracyₘ − majority_class_rate , 0 )

Cross-validation happens on the **training** split only, so the test numbers
stay honest.

### Setup on Kaggle
1. **Add Input** → `masoudnickparvar/brain-tumor-mri-dataset`
2. **Add Input** → `paultimothymooney/chest-xray-pneumonia`
3. **Settings → Accelerator → GPU T4 x2**
4. **Run All** (~10 min per dataset)

> **Use `masoudnickparvar/brain-tumor-mri-dataset`, not `sartajbhuvaji/…`.**
> The Sartaj version has a well-known defect: its `Testing/glioma_tumor` folder
> holds images from different sources than `Training/glioma_tumor` — uniformly
> 512×512 in training, but a mix of 236×236, 524×581, 554×554 and others in
> testing. A model trained on it classifies training gliomas 20/20 and testing
> gliomas **1/20**, while scoring 95% on the other three classes. That is
> measuring the dataset, not the model. The masoudnickparvar release is the
> cleaned, re-labelled version (7,023 images) and does not have the problem.
> This notebook works with either — the paths and class names are discovered.

Produces, per task, `<task>_backbone.keras` and `<task>_heads.joblib` for the
repo's `models/image/` folder.

In [ ]:
import os, json, time, numpy as np, pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import joblib

from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import (ExtraTreesClassifier, GradientBoostingClassifier,
                              RandomForestClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, f1_score, roc_auc_score)
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

SEED = 42
np.random.seed(SEED); tf.random.set_seed(SEED)
AUTOTUNE = tf.data.AUTOTUNE
IMG_SIZE, BATCH = 224, 32

print("TensorFlow:", tf.__version__, "| XGBoost:", HAS_XGB)
gpus = tf.config.list_physical_devices("GPU")
print("GPUs:", gpus if gpus else "NONE - set Settings > Accelerator > GPU T4 x2")

## 1. Find whichever datasets are attached

Kaggle mirrors nest their folders differently, so the paths are discovered
rather than hardcoded. Each dataset is identified by its class folder names.

In [ ]:
def find_split_dirs(root="/kaggle/input", max_depth=5):
    """Every (train_dir, test_dir) pair under /kaggle/input."""
    root = Path(root)
    if not root.exists():
        return []
    found = []
    for dirpath, dirnames, _ in os.walk(root):
        rel = Path(dirpath).relative_to(root)
        if len(rel.parts) > max_depth:
            dirnames[:] = []
            continue
        here = Path(dirpath)
        names = {d.name.lower(): d for d in here.iterdir() if d.is_dir()}
        train = names.get("training") or names.get("train")
        test = names.get("testing") or names.get("test")
        if train is not None and any(c.is_dir() for c in train.iterdir()):
            found.append((train, test))
    return found

def identify(train_dir):
    """Name the task from its class folders."""
    classes = sorted(d.name for d in train_dir.iterdir() if d.is_dir())
    joined = " ".join(classes).lower()
    if "pneumonia" in joined:
        return "pneumonia", classes
    if "glioma" in joined or "tumor" in joined or "tumour" in joined:
        return "brain_tumor", classes
    return None, classes


def warn_if_split_mismatch(name, train_dir, test_dir, sample=40):
    """Flag classes whose train and test images look physically different.

    The Sartaj brain-MRI dataset ships a Testing/glioma_tumor folder sourced
    differently from Training/glioma_tumor, which silently destroys that
    class's test score. Comparing image dimensions catches it early.
    """
    if test_dir is None:
        return
    from collections import Counter
    from PIL import Image
    for cls in sorted(d.name for d in train_dir.iterdir() if d.is_dir()):
        if not (test_dir / cls).exists():
            continue
        dims = {}
        for split, folder in (("train", train_dir / cls), ("test", test_dir / cls)):
            c = Counter()
            for p in sorted(folder.glob("*"))[:sample]:
                try:
                    with Image.open(p) as im:
                        c[im.size] += 1
                except Exception:
                    pass
            dims[split] = c
        if not dims["train"] or not dims["test"]:
            continue
        tr_share = dims["train"].most_common(1)[0][1] / sum(dims["train"].values())
        te_share = dims["test"].most_common(1)[0][1] / sum(dims["test"].values())
        if tr_share > 0.9 and te_share < 0.5:
            print(f"   !! {name}/{cls}: train images are uniformly "
                  f"{dims['train'].most_common(1)[0][0]} but test images are "
                  f"mixed {dims['test'].most_common(3)}")
            print(f"      the two folders may not hold comparable images; "
                  f"treat this class's test score with suspicion")

TASKS = {}
for train_dir, test_dir in find_split_dirs():
    name, classes = identify(train_dir)
    if name is None or name in TASKS:
        continue
    TASKS[name] = {"train": train_dir, "test": test_dir, "classes": classes}

if not TASKS:
    raise SystemExit("No dataset found. Add Input -> brain-tumor-classification-mri "
                     "and/or chest-xray-pneumonia.")

for name, t in TASKS.items():
    print(f"{name}")
    print(f"   train: {t['train']}")
    print(f"   test : {t['test']}")
    for c in t["classes"]:
        n_tr = len(list((t["train"] / c).glob("*")))
        n_te = len(list((t["test"] / c).glob("*"))) if t["test"] and (t["test"] / c).exists() else 0
        print(f"      {c:<20} train {n_tr:>5}   test {n_te:>5}")
    warn_if_split_mismatch(name, t["train"], t["test"])

## 2. The ten algorithms

Exactly the roster used for the tabular diseases, so the two halves of the project are directly comparable.

In [ ]:
def build_models(seed=SEED):
    models = {
        "Logistic Regression": LogisticRegression(max_iter=3000, C=1.0, random_state=seed),
        "Gaussian Naive Bayes": GaussianNB(),
        "K-Nearest Neighbours": KNeighborsClassifier(n_neighbors=9, weights="distance"),
        "Decision Tree": DecisionTreeClassifier(max_depth=8, min_samples_leaf=5, random_state=seed),
        "Random Forest": RandomForestClassifier(n_estimators=400, min_samples_leaf=2,
                                                n_jobs=-1, random_state=seed),
        "Extra Trees": ExtraTreesClassifier(n_estimators=400, min_samples_leaf=2,
                                            n_jobs=-1, random_state=seed),
        # kept modest on purpose: boosting over 128 PCA components is by far the
        # slowest member here (~5 min of the run), and more trees buy little
        "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, learning_rate=0.1,
                                                        max_depth=3, random_state=seed),
        "Support Vector Machine": SVC(C=10.0, kernel="rbf", gamma="scale",
                                      probability=True, random_state=seed),
        "Neural Network (MLP)": MLPClassifier(hidden_layer_sizes=(256, 128), alpha=1e-3,
                                              max_iter=1200, early_stopping=True,
                                              n_iter_no_change=25, random_state=seed),
    }
    if HAS_XGB:
        models["XGBoost"] = XGBClassifier(n_estimators=400, learning_rate=0.06, max_depth=4,
                                          subsample=0.9, colsample_bytree=0.9,
                                          eval_metric="mlogloss", random_state=seed,
                                          n_jobs=-1, tree_method="hist")
    else:
        models["Linear Discriminant Analysis"] = LinearDiscriminantAnalysis()
    return models

print(f"{len(build_models())} algorithms:")
for i, n in enumerate(build_models(), 1):
    print(f"  {i:>2}. {n}")

## 3. Backbone: fine-tune, then extract embeddings

Two stages. The frozen stage lets the new head settle without wrecking the
pretrained filters; the fine-tune stage unfreezes the top of the backbone at a
much lower learning rate so the features adapt to medical imagery. Afterwards
the classification head is discarded and the pooled 1280-d layer becomes the
feature extractor for the ten algorithms.

In [ ]:
def make_datasets(task, seed=SEED):
    classes = task["classes"]
    common = dict(image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH,
                  label_mode="int", class_names=classes)
    train = tf.keras.utils.image_dataset_from_directory(
        str(task["train"]), validation_split=0.2, subset="training", seed=seed, **common)
    val = tf.keras.utils.image_dataset_from_directory(
        str(task["train"]), validation_split=0.2, subset="validation", seed=seed, **common)
    if task["test"] is not None:
        test = tf.keras.utils.image_dataset_from_directory(
            str(task["test"]), shuffle=False, **common)
    else:
        print("   no test folder; reusing the validation split for reporting")
        test = tf.keras.utils.image_dataset_from_directory(
            str(task["train"]), validation_split=0.2, subset="validation",
            seed=seed, shuffle=False, **common)
    return (train.prefetch(AUTOTUNE), val.prefetch(AUTOTUNE), test.prefetch(AUTOTUNE))


def train_backbone(train_ds, val_ds, n_classes, frozen_epochs=12, ft_epochs=8):
    augment = tf.keras.Sequential([
        tf.keras.layers.RandomFlip("horizontal"),
        tf.keras.layers.RandomRotation(0.06),
        tf.keras.layers.RandomZoom(0.10),
        tf.keras.layers.RandomContrast(0.10),
    ], name="augment")

    base = tf.keras.applications.EfficientNetB0(
        input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights="imagenet")
    base.trainable = False

    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = augment(inputs)
    x = tf.keras.applications.efficientnet.preprocess_input(x)
    x = base(x, training=False)
    embedding = tf.keras.layers.GlobalAveragePooling2D(name="embedding")(x)
    x = tf.keras.layers.Dropout(0.3)(embedding)
    outputs = tf.keras.layers.Dense(n_classes, activation="softmax")(x)
    model = tf.keras.Model(inputs, outputs)

    cb = [tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=5,
                                           restore_best_weights=True),
          tf.keras.callbacks.ReduceLROnPlateau(patience=3, factor=0.5, min_lr=1e-6)]

    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    print("   stage 1: frozen backbone")
    h1 = model.fit(train_ds, validation_data=val_ds, epochs=frozen_epochs,
                   callbacks=cb, verbose=2)

    base.trainable = True
    for layer in base.layers[:-60]:
        layer.trainable = False
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    print("   stage 2: fine-tuning the top of the backbone")
    h2 = model.fit(train_ds, validation_data=val_ds, epochs=ft_epochs,
                   callbacks=cb, verbose=2)

    extractor = tf.keras.Model(model.input, model.get_layer("embedding").output)
    return model, extractor, (h1, h2)


def embed(extractor, ds):
    X = extractor.predict(ds, verbose=0)
    y = np.concatenate([lab.numpy() for _, lab in ds])
    return X, y

## 4. The weighted ensemble

`weight = max(cv_accuracy − majority_class_rate, 0)`. A model that merely
matches the "always guess the commonest class" baseline has learned nothing and
gets a weight of zero, so it cannot drag the vote.

In [ ]:
def fit_weighted_ensemble(X_train, y_train, n_components=128, cv_folds=5):
    counts = np.bincount(y_train)
    majority = counts.max() / counts.sum()
    n_comp = int(min(n_components, X_train.shape[1], len(X_train) - 1))
    cv = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=SEED)

    fitted, scores = {}, {}
    print(f"   {'model':<28}{'CV acc':>9}{'weight':>9}")
    print("   " + "-" * 46)
    for name, est in build_models().items():
        pipe = Pipeline([("scale", StandardScaler()),
                         ("pca", PCA(n_components=n_comp, random_state=SEED)),
                         ("clf", est)])
        proba = cross_val_predict(pipe, X_train, y_train, cv=cv,
                                  method="predict_proba", n_jobs=1)
        acc = accuracy_score(y_train, proba.argmax(1))
        weight = max(acc - majority, 0.0)
        pipe.fit(X_train, y_train)
        fitted[name] = pipe
        scores[name] = {"cv_accuracy": float(acc), "weight_raw": float(weight),
                        "cv_f1_macro": float(f1_score(y_train, proba.argmax(1),
                                                      average="macro"))}
        print(f"   {name:<28}{acc:>9.4f}{weight:>9.4f}")

    total = sum(s["weight_raw"] for s in scores.values())
    for s in scores.values():
        s["weight"] = s["weight_raw"] / total if total > 0 else 1.0 / len(scores)
    return fitted, scores, float(majority), n_comp


def ensemble_proba(fitted, scores, X):
    out = None
    for name, model in fitted.items():
        w = scores[name]["weight"]
        if w <= 0:
            continue
        p = model.predict_proba(X) * w
        out = p if out is None else out + p
    return out

## 5. Train every attached task

In [ ]:
RESULTS = {}

for task_name, task in TASKS.items():
    print("=" * 66)
    print(task_name.upper())
    print("=" * 66)
    t0 = time.time()
    classes = task["classes"]

    train_ds, val_ds, test_ds = make_datasets(task)
    model, extractor, hists = train_backbone(train_ds, val_ds, len(classes))

    print("   extracting embeddings ...")
    # rebuild train without shuffling so features and labels stay aligned
    train_eval = tf.keras.utils.image_dataset_from_directory(
        str(task["train"]), validation_split=0.2, subset="training", seed=SEED,
        shuffle=False, image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH,
        label_mode="int", class_names=classes).prefetch(AUTOTUNE)
    X_tr, y_tr = embed(extractor, train_eval)
    X_te, y_te = embed(extractor, test_ds)
    print(f"   embeddings: train {X_tr.shape}  test {X_te.shape}")

    fitted, scores, majority, n_comp = fit_weighted_ensemble(X_tr, y_tr)

    # ---- held-out test: every member, then the ensemble
    print(f"\n   held-out test ({len(y_te)} images)")
    print(f"   {'model':<28}{'acc':>9}{'macro F1':>10}")
    print("   " + "-" * 48)
    member_test = {}
    for name, m in fitted.items():
        p = m.predict_proba(X_te)
        pred = p.argmax(1)
        member_test[name] = {"accuracy": float(accuracy_score(y_te, pred)),
                             "f1_macro": float(f1_score(y_te, pred, average="macro"))}
        print(f"   {name:<28}{member_test[name]['accuracy']:>9.4f}"
              f"{member_test[name]['f1_macro']:>10.4f}")

    ens_p = ensemble_proba(fitted, scores, X_te)
    ens_pred = ens_p.argmax(1)
    ens_acc = accuracy_score(y_te, ens_pred)
    ens_f1 = f1_score(y_te, ens_pred, average="macro")
    print("   " + "-" * 48)
    print(f"   {'>> WEIGHTED ENSEMBLE':<28}{ens_acc:>9.4f}{ens_f1:>10.4f}")

    # the honest comparison: the model you would have picked from CV alone
    cv_pick = max(scores, key=lambda n: scores[n]["cv_accuracy"])
    print(f"\n   model picked by CV on train: {cv_pick} "
          f"(test {member_test[cv_pick]['accuracy']:.4f})")
    print(f"   ensemble delta: {ens_acc - member_test[cv_pick]['accuracy']:+.4f}")

    cnn_loss, cnn_acc = model.evaluate(test_ds, verbose=0)
    print(f"   plain fine-tuned CNN for reference: {cnn_acc:.4f}")

    print(classification_report(y_te, ens_pred, target_names=classes, digits=4))

    cm = confusion_matrix(y_te, ens_pred)
    plt.figure(figsize=(1.6 * len(classes) + 3, 1.3 * len(classes) + 2.5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=classes, yticklabels=classes, cbar=False)
    plt.title(f"{task_name} — weighted ensemble, accuracy {ens_acc:.2%}", weight="bold")
    plt.xlabel("Predicted"); plt.ylabel("True"); plt.tight_layout(); plt.show()

    # ---- save
    extractor.save(f"/kaggle/working/{task_name}_backbone.keras")
    joblib.dump({"task": task_name, "classes": classes, "img_size": IMG_SIZE,
                 "backbone": "EfficientNetB0", "pca_components": n_comp,
                 "members": fitted, "scores": scores, "majority_rate": majority},
                f"/kaggle/working/{task_name}_heads.joblib", compress=3)

    RESULTS[task_name] = {
        "classes": classes, "n_train": int(len(y_tr)), "n_test": int(len(y_te)),
        "majority_rate": majority,
        "cv_scores": {k: v for k, v in scores.items()},
        "test_members": member_test,
        "test_ensemble": {"accuracy": float(ens_acc), "f1_macro": float(ens_f1)},
        "cv_selected_model": {"name": cv_pick,
                              "test_accuracy": member_test[cv_pick]["accuracy"]},
        "plain_cnn_test_accuracy": float(cnn_acc),
        "confusion_matrix": cm.tolist(),
        "minutes": round((time.time() - t0) / 60, 1),
    }
    print(f"   done in {RESULTS[task_name]['minutes']} min\n")

## 6. Summary and downloads

In [ ]:
rows = []
for name, r in RESULTS.items():
    rows.append({
        "task": name,
        "classes": len(r["classes"]),
        "test images": r["n_test"],
        "baseline": round(r["majority_rate"], 4),
        "ensemble acc": round(r["test_ensemble"]["accuracy"], 4),
        "ensemble F1": round(r["test_ensemble"]["f1_macro"], 4),
        "CV-picked model": r["cv_selected_model"]["name"],
        "its acc": round(r["cv_selected_model"]["test_accuracy"], 4),
        "plain CNN": round(r["plain_cnn_test_accuracy"], 4),
    })
print(pd.DataFrame(rows).to_string(index=False))

with open("/kaggle/working/image_ensemble_metrics.json", "w") as f:
    json.dump(RESULTS, f, indent=2)

print("\nFiles to download from the Output panel:")
for f in sorted(os.listdir("/kaggle/working")):
    size = os.path.getsize(f"/kaggle/working/{f}") / 1e6
    print(f"   {f:<38} {size:>7.1f} MB")

In [ ]:
# weight breakdown per task — which algorithms actually carry the vote
for name, r in RESULTS.items():
    df = pd.DataFrame([
        {"algorithm": k, "CV accuracy %": 100 * v["cv_accuracy"],
         "weight %": 100 * v["weight"],
         "test accuracy %": 100 * r["test_members"][k]["accuracy"]}
        for k, v in r["cv_scores"].items()
    ]).sort_values("weight %", ascending=False)
    print(f"\n{name}")
    print(df.to_string(index=False, float_format=lambda v: f"{v:.2f}"))

---

## Putting the models back in the repo

Download these into `02-medical-disease-detection/models/image/`:

| File | What it is |
|---|---|
| `brain_tumor_backbone.keras` | fine-tuned EfficientNetB0 feature extractor |
| `brain_tumor_heads.joblib` | the 10 fitted classifiers + their weights |
| `pneumonia_backbone.keras` | same, for chest X-rays |
| `pneumonia_heads.joblib` | same, for chest X-rays |

and `image_ensemble_metrics.json` into `reports/`.

`streamlit run app.py` then shows the same 10-algorithm breakdown for an
uploaded scan as it does for the tabular diseases — each model's vote, its
surety, its accuracy weight, and the combined verdict.